In [ ]:
import pandas as pd

In [ ]:
q12024 = pd.read_excel('2024comparendosQ1.xlsx', header=9)
q22024 = pd.read_excel('2024comparendosQ2.xlsx', header=9)
q32024 = pd.read_excel('2024comparendosQ3.xlsx', header=9)
q42024 = pd.read_excel('2024comparendosQ4.xlsx', header=9)

In [ ]:
q12024['TRIMESTRE'] = 1

In [ ]:
q22024['TRIMESTRE'] = 2
q32024['TRIMESTRE'] = 3
q42024['TRIMESTRE'] = 4

In [ ]:
q22024.info()

In [ ]:
df2024total = pd.concat([q12024, q22024, q32024, q42024], ignore_index=True)

In [ ]:
df2024total.info()

In [ ]:
totalcomparendos2024 = df2024total['CANTIDAD'].sum()
totalcomparendos2024

In [ ]:
filepath = 'PPED-AreaMun-2018-2042VP (1).xlsx'

df = pd.read_excel(filepath, sheet_name='PobMunicipalxÁrea', skiprows=7)
df = df.dropna(subset=['AÑO', 'ÁREA GEOGRÁFICA'])

df2024total_pob = df[(df['AÑO'] == 2024) & (df['ÁREA GEOGRÁFICA'] == 'Total')].copy()
dfresultado = df2024total_pob[['DP', 'DPNOM', 'MPIO', 'DPMP', 'TOTAL']]

dfresultado.columns = ['CódigoDepartamento', 'Departamento', 'CÓDIGO DIVIPOLA', 'MUNICIPIO', 'PoblacionTotal2024']
dfresultado = dfresultado.reset_index(drop=True)

outputpath = 'Poblacion2024TotalMunicipios.xlsx'
dfresultado.to_excel(outputpath, index=False)

print(f'Total de municipios extraídos: {len(dfresultado)}')
print('\nPrimeros 10 registros:')
dfresultado.head(10)
print('\nEstadísticas básicas:')
dfresultado['PoblacionTotal2024'].describe()

In [ ]:
poblacion2024 = dfresultado
poblacion2024

In [ ]:
dfmerge = poblacion2024[['CÓDIGO DIVIPOLA', 'Departamento', 'MUNICIPIO', 'PoblacionTotal2024']].copy()

dfmerge['CÓDIGO DIVIPOLA'] = pd.to_numeric(dfmerge['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')
df2024total['CÓDIGO DIVIPOLA'] = pd.to_numeric(df2024total['CÓDIGO DIVIPOLA'], errors='coerce').astype('Int64')

df2024total = df2024total.merge(
    dfmerge,
    on='CÓDIGO DIVIPOLA',
    how='left',
    suffixes=('', '_pob')
)

df2024total['DEPARTAMENTO'] = df2024total['Departamento'].fillna(df2024total['DEPARTAMENTO'])
df2024total['MUNICIPIO'] = df2024total['MUNICIPIO_pob'].fillna(df2024total['MUNICIPIO'])

df2024total = df2024total.rename(columns={'PoblacionTotal2024': 'PoblacionTotalMunicipio2024'})
df2024total = df2024total.drop(columns=['Departamento', 'MUNICIPIO_pob'])

In [ ]:
df2024total.info()

In [ ]:
df2024total.head()

In [ ]:
municipio2024 = (
    df2024total
    .groupby(['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO'], as_index=False)
    .agg(
        comparendos=('CANTIDAD', 'sum'),
        poblacion=('PoblacionTotalMunicipio2024', 'max')
    )
)

municipio2024['poblacion'] = municipio2024['poblacion'].replace(0, pd.NA)
municipio2024['tasacomparendos100k'] = (
    municipio2024['comparendos'] / municipio2024['poblacion'] * 100000
)

totalcomparendos_calc = municipio2024['comparendos'].sum()

print(f'Total comparendos calculado desde agrupación municipal: {totalcomparendos_calc}')
print(f'totalcomparendos2024: {totalcomparendos2024}')
print(f'Coincide: {totalcomparendos_calc == totalcomparendos2024}')
print(municipio2024.sort_values('tasacomparendos100k', ascending=False).head(10))

In [ ]:
df2024total = df2024total.merge(
    municipio2024[['CÓDIGO DIVIPOLA', 'tasacomparendos100k']],
    on='CÓDIGO DIVIPOLA',
    how='left'
)

df2024total[['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO', 'CANTIDAD', 'tasacomparendos100k']].head()

In [ ]:
df2024total.info()

In [ ]:
municipios_unicos = df2024total[['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO']].drop_duplicates()
municipios_unicos = municipios_unicos.sort_values('CÓDIGO DIVIPOLA').reset_index(drop=True)
print(municipios_unicos)

In [ ]:
def clasificar_categoria(pob):
    if pob > 500001:
        return 'Categoría especial'
    elif 100001 < pob <= 500000:
        return 'Categoría 1'
    elif 50001 < pob <= 100000:
        return 'Categoría 2'
    elif 30001 < pob <= 50000:
        return 'Categoría 3'
    elif 20001 < pob <= 30000:
        return 'Categoría 4'
    elif 10001 < pob <= 20000:
        return 'Categoría 5'
    else:
        return 'Categoría 6'

df2024total['categoria_municipio'] = df2024total['PoblacionTotalMunicipio2024'].apply(clasificar_categoria)

In [ ]:
df2024total.to_csv('comparendos2024conpoblacion.csv', index=False)

In [ ]:
print('Top 10 municipios por tasa de comparendos por 100k habitantes')
top10tasas = municipio2024.sort_values('tasacomparendos100k', ascending=False).head(10)
print(top10tasas[['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO', 'comparendos', 'poblacion', 'tasacomparendos100k']])

In [ ]:
dfresultado_muni = dfresultado[['CÓDIGO DIVIPOLA', 'Departamento', 'MUNICIPIO']].drop_duplicates().rename(columns={'Departamento': 'DEPARTAMENTO'})

municipios_con_comparendos = df2024total[['CÓDIGO DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO']].drop_duplicates()

faltantes_municipios = dfresultado_muni[~dfresultado_muni['CÓDIGO DIVIPOLA'].isin(municipios_con_comparendos['CÓDIGO DIVIPOLA'])].copy()
faltantes_municipios = faltantes_municipios.sort_values('CÓDIGO DIVIPOLA').reset_index(drop=True)

print('Municipios totales población:', len(dfresultado_muni))
print('Municipios totales comparendos:', len(municipios_con_comparendos))
print('Municipios faltantes:', len(faltantes_municipios))

faltantes_municipios.head(20)

In [ ]:
municipio2024.to_excel('municipios2024.xlsx', index=False)
faltantes_municipios.to_excel('faltantes_municipios.xlsx', index=False)